# 🛒 UK Retail Sales Analysis with SQL & Python
### Relational Database Design, SQL Querying & Business Insight Visualisation

**Author:** Kaveen Kodikarage

---

## Project Overview

This project demonstrates how to design a **relational database**, populate it with realistic UK retail data, and extract business insights using **SQL queries of increasing complexity** - from basic aggregations through to window functions and CTEs. Results are visualised using Python.

### Skills Demonstrated:
- Relational database schema design (normalisation, foreign keys)
- SQLite database creation and population via Python
- SQL: `SELECT`, `WHERE`, `GROUP BY`, `ORDER BY`, `JOIN`
- SQL: Subqueries, `HAVING`, `CASE WHEN`
- SQL: Window functions (`RANK`, `ROW_NUMBER`, running totals)
- SQL: Common Table Expressions (CTEs)
- Data visualisation with Matplotlib & Seaborn

### Database Schema:
```
customers --< orders --< order_items >--products
                │
           (ship_mode, dates)
```

---

## 1. Setup & Imports

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import random
from datetime import date, timedelta
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.family'] = 'DejaVu Sans'

NAVY  = '#1B3A6B'
BLUE  = '#2E5FA3'
LIGHT = '#7AAAD8'
GREEN = '#27AE60'
RED   = '#C0392B'

random.seed(42)
np.random.seed(42)

print('Setup complete.')

## 2. Database Schema Design

We create four related tables following **3rd Normal Form (3NF)** — a standard used in professional database design to reduce redundancy and ensure data integrity.

- **customers** — who bought
- **products** — what was sold
- **orders** — the transaction header
- **order_items** — individual line items (links orders ↔ products)

In [ ]:
# Create in-memory SQLite database (swap ':memory:' for 'retail.db' to persist)
conn = sqlite3.connect('uk_retail.db')
cur  = conn.cursor()

# Drop tables if re-running
for tbl in ['order_items', 'orders', 'customers', 'products']:
    cur.execute(f'DROP TABLE IF EXISTS {tbl}')

# ── CUSTOMERS ────────────────────────────────────────────────────────────────
cur.execute('''
CREATE TABLE customers (
    customer_id   INTEGER PRIMARY KEY,
    first_name    TEXT    NOT NULL,
    last_name     TEXT    NOT NULL,
    city          TEXT    NOT NULL,
    region        TEXT    NOT NULL,
    segment       TEXT    NOT NULL   -- Consumer / Corporate / Home Office
)
''')

# ── PRODUCTS ─────────────────────────────────────────────────────────────────
cur.execute('''
CREATE TABLE products (
    product_id    INTEGER PRIMARY KEY,
    product_name  TEXT    NOT NULL,
    category      TEXT    NOT NULL,
    sub_category  TEXT    NOT NULL,
    unit_price    REAL    NOT NULL
)
''')

# ── ORDERS ───────────────────────────────────────────────────────────────────
cur.execute('''
CREATE TABLE orders (
    order_id      INTEGER PRIMARY KEY,
    customer_id   INTEGER NOT NULL,
    order_date    TEXT    NOT NULL,
    ship_date     TEXT    NOT NULL,
    ship_mode     TEXT    NOT NULL,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
)
''')

# ── ORDER_ITEMS ──────────────────────────────────────────────────────────────
cur.execute('''
CREATE TABLE order_items (
    item_id       INTEGER PRIMARY KEY,
    order_id      INTEGER NOT NULL,
    product_id    INTEGER NOT NULL,
    quantity      INTEGER NOT NULL,
    discount      REAL    NOT NULL DEFAULT 0,
    sales         REAL    NOT NULL,
    profit        REAL    NOT NULL,
    FOREIGN KEY (order_id)   REFERENCES orders(order_id),
    FOREIGN KEY (product_id) REFERENCES products(product_id)
)
''')

conn.commit()
print('Schema created: customers, products, orders, order_items')

## 3. Populating the Database with Realistic UK Retail Data

In [ ]:
# ── Reference data ────────────────────────────────────────────────────────────
first_names = ['James','Olivia','Mohammed','Sophie','Daniel','Emma','Liam','Amelia',
               'Harry','Grace','Oliver','Charlotte','Noah','Isla','George','Poppy',
               'Jack','Lily','Ethan','Ava','William','Freya','Thomas','Alice']
last_names  = ['Smith','Jones','Williams','Taylor','Brown','Davies','Evans','Wilson',
               'Thomas','Roberts','Johnson','Walker','Wright','Thompson','White','Hughes']

regions_cities = {
    'London':        ['Camden','Hackney','Southwark','Wandsworth','Islington'],
    'South East':    ['Reading','Brighton','Oxford','Southampton','Guildford'],
    'North West':    ['Manchester','Liverpool','Preston','Salford','Bolton'],
    'Yorkshire':     ['Leeds','Sheffield','Bradford','Hull','York'],
    'South West':    ['Bristol','Exeter','Plymouth','Bath','Swindon'],
    'East Midlands': ['Nottingham','Leicester','Derby','Lincoln','Coventry'],
    'Scotland':      ['Edinburgh','Glasgow','Aberdeen','Dundee','Inverness'],
    'Wales':         ['Cardiff','Swansea','Newport','Wrexham','Barry'],
}
segments = ['Consumer', 'Corporate', 'Home Office']

products_data = [
    # (name, category, sub_category, unit_price)
    ('Staples Pack (200)', 'Office Supplies', 'Fasteners', 2.49),
    ('A4 Copy Paper (500 sheets)', 'Office Supplies', 'Paper', 5.99),
    ('Ballpoint Pens (Pack of 10)', 'Office Supplies', 'Pens', 3.29),
    ('Lever Arch File A4', 'Office Supplies', 'Binders', 4.79),
    ('Sticky Notes 100pk', 'Office Supplies', 'Paper', 2.99),
    ('Whiteboard Markers', 'Office Supplies', 'Pens', 6.49),
    ('Desk Organiser', 'Office Supplies', 'Storage', 12.99),
    ('Scissors (Professional)', 'Office Supplies', 'Fasteners', 7.49),
    ('HP LaserJet Printer', 'Technology', 'Machines', 299.99),
    ('Lenovo ThinkPad Laptop', 'Technology', 'Machines', 849.00),
    ('27-inch Monitor', 'Technology', 'Machines', 379.00),
    ('Wireless Keyboard & Mouse', 'Technology', 'Accessories', 45.00),
    ('USB-C Hub (7-port)', 'Technology', 'Accessories', 29.99),
    ('External SSD 1TB', 'Technology', 'Accessories', 89.99),
    ('Webcam HD 1080p', 'Technology', 'Accessories', 54.99),
    ('Smartphone Stand', 'Technology', 'Accessories', 14.99),
    ('Printer Ink Cartridges', 'Technology', 'Accessories', 24.99),
    ('Standing Desk (Electric)', 'Furniture', 'Desks', 499.00),
    ('Ergonomic Office Chair', 'Furniture', 'Chairs', 349.00),
    ('3-Drawer Filing Cabinet', 'Furniture', 'Storage', 189.00),
    ('Adjustable Monitor Stand', 'Furniture', 'Desks', 39.99),
    ('Bookcase (5-shelf)', 'Furniture', 'Bookcases', 129.00),
    ('Folding Table', 'Furniture', 'Tables', 79.99),
    ('Meeting Room Table', 'Furniture', 'Tables', 599.00),
    ('Visitor Chair (Set of 2)', 'Furniture', 'Chairs', 159.00),
]

ship_modes = ['Standard Class', 'Second Class', 'First Class', 'Same Day']
ship_days  = {'Standard Class': 5, 'Second Class': 3, 'First Class': 2, 'Same Day': 0}

# ── Insert products ───────────────────────────────────────────────────────────
cur.executemany(
    'INSERT INTO products (product_name, category, sub_category, unit_price) VALUES (?,?,?,?)',
    products_data
)

# ── Insert customers ──────────────────────────────────────────────────────────
NUM_CUSTOMERS = 300
customers_rows = []
for i in range(1, NUM_CUSTOMERS + 1):
    region = random.choice(list(regions_cities.keys()))
    city   = random.choice(regions_cities[region])
    customers_rows.append((
        random.choice(first_names),
        random.choice(last_names),
        city, region,
        random.choice(segments)
    ))
cur.executemany(
    'INSERT INTO customers (first_name, last_name, city, region, segment) VALUES (?,?,?,?,?)',
    customers_rows
)

# ── Insert orders + order_items ───────────────────────────────────────────────
start_date = date(2021, 1, 1)
end_date   = date(2023, 12, 31)
date_range = (end_date - start_date).days

cur.execute('SELECT product_id, unit_price, category FROM products')
all_products = cur.fetchall()

item_id  = 1
NUM_ORDERS = 3000

for order_id in range(1, NUM_ORDERS + 1):
    customer_id = random.randint(1, NUM_CUSTOMERS)
    ship_mode   = random.choice(ship_modes)
    order_date  = start_date + timedelta(days=random.randint(0, date_range))
    ship_date   = order_date + timedelta(days=ship_days[ship_mode] + random.randint(0, 2))

    cur.execute(
        'INSERT INTO orders (order_id, customer_id, order_date, ship_date, ship_mode) VALUES (?,?,?,?,?)',
        (order_id, customer_id, str(order_date), str(ship_date), ship_mode)
    )

    # 1–4 line items per order
    num_items = random.randint(1, 4)
    chosen_products = random.sample(all_products, min(num_items, len(all_products)))

    for prod_id, unit_price, category in chosen_products:
        qty      = random.randint(1, 10)
        discount = random.choice([0, 0, 0, 0.1, 0.2, 0.3])
        sales    = round(qty * unit_price * (1 - discount), 2)
        # Profit margin varies by category
        margin   = {'Office Supplies': 0.35, 'Technology': 0.18, 'Furniture': 0.12}.get(category, 0.2)
        profit   = round(sales * margin * random.uniform(0.5, 1.5), 2)
        cur.execute(
            'INSERT INTO order_items (item_id, order_id, product_id, quantity, discount, sales, profit) VALUES (?,?,?,?,?,?,?)',
            (item_id, order_id, prod_id, qty, discount, sales, profit)
        )
        item_id += 1

conn.commit()

# Summary
for tbl in ['customers', 'products', 'orders', 'order_items']:
    cur.execute(f'SELECT COUNT(*) FROM {tbl}')
    print(f'  {tbl:<15} {cur.fetchone()[0]:>6,} rows')

---
## 4. SQL Analysis

### Helper function
We use a helper that runs a SQL query and returns a pandas DataFrame — a common pattern in professional data work.

In [ ]:
def query(sql, label=''):
    """Run SQL and return a DataFrame. Optionally print results."""
    df = pd.read_sql_query(sql, conn)
    if label:
        print(f'=== {label} ===')
        print(df.to_string(index=False))
        print()
    return df

### Q1 — Total Revenue, Profit and Orders by Year
*Basic aggregation with GROUP BY and ORDER BY*

In [ ]:
sql_q1 = '''
SELECT
    strftime('%Y', o.order_date)   AS year,
    COUNT(DISTINCT o.order_id)     AS total_orders,
    ROUND(SUM(oi.sales), 2)        AS total_revenue,
    ROUND(SUM(oi.profit), 2)       AS total_profit,
    ROUND(SUM(oi.profit) * 100.0
          / SUM(oi.sales), 1)      AS profit_margin_pct
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
GROUP BY year
ORDER BY year
'''
q1 = query(sql_q1, 'Revenue by Year')

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()
ax1.bar(q1['year'], q1['total_revenue'], color=NAVY, label='Revenue', width=0.4)
ax1.bar([str(int(y)+0.4) for y in q1['year']], q1['total_profit'],
        color=GREEN, label='Profit', width=0.4)
ax2.plot(q1['year'], q1['profit_margin_pct'], color=RED, marker='o',
         linewidth=2, label='Profit Margin %')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'£{x/1000:.0f}k'))
ax2.set_ylabel('Profit Margin %', color=RED)
ax1.set_ylabel('£ Value')
ax1.set_title('Annual Revenue, Profit & Margin', fontsize=13, fontweight='bold')
lines1, l1 = ax1.get_legend_handles_labels()
lines2, l2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2, l1+l2, loc='upper left')
plt.tight_layout()
plt.savefig('q1_annual_revenue.png', dpi=150, bbox_inches='tight')
plt.show()

### Q2 — Revenue and Profit by Product Category
*GROUP BY with computed profit margin column*

In [ ]:
sql_q2 = '''
SELECT
    p.category,
    COUNT(DISTINCT o.order_id)              AS orders,
    ROUND(SUM(oi.sales), 2)                 AS revenue,
    ROUND(SUM(oi.profit), 2)                AS profit,
    ROUND(AVG(oi.discount) * 100, 1)        AS avg_discount_pct,
    ROUND(SUM(oi.profit)*100.0/SUM(oi.sales),1) AS margin_pct
FROM order_items oi
JOIN orders   o ON oi.order_id  = o.order_id
JOIN products p ON oi.product_id = p.product_id
GROUP BY p.category
ORDER BY revenue DESC
'''
q2 = query(sql_q2, 'Revenue by Category')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colours = [NAVY, BLUE, LIGHT]
axes[0].bar(q2['category'], q2['revenue'], color=colours)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'£{x/1000:.0f}k'))
axes[0].set_title('Revenue by Category', fontweight='bold')

axes[1].bar(q2['category'], q2['margin_pct'], color=colours)
axes[1].set_ylabel('Profit Margin %')
axes[1].set_title('Profit Margin % by Category', fontweight='bold')
for ax in axes:
    ax.set_xlabel('')
plt.suptitle('Category Performance', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('q2_category_performance.png', dpi=150, bbox_inches='tight')
plt.show()

### Q3 — Top 10 Customers by Revenue
*Multi-table JOIN with string concatenation and LIMIT*

In [ ]:
sql_q3 = '''
SELECT
    c.customer_id,
    c.first_name || ' ' || c.last_name  AS customer_name,
    c.region,
    c.segment,
    COUNT(DISTINCT o.order_id)           AS total_orders,
    ROUND(SUM(oi.sales), 2)              AS lifetime_value,
    ROUND(SUM(oi.profit), 2)             AS total_profit
FROM customers c
JOIN orders      o  ON c.customer_id  = o.customer_id
JOIN order_items oi ON o.order_id     = oi.order_id
GROUP BY c.customer_id
ORDER BY lifetime_value DESC
LIMIT 10
'''
q3 = query(sql_q3, 'Top 10 Customers by Revenue')

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(q3['customer_name'][::-1], q3['lifetime_value'][::-1],
               color=NAVY, edgecolor='white')
for bar, (_, row) in zip(bars, q3[::-1].iterrows()):
    ax.text(bar.get_width() + 200, bar.get_y() + bar.get_height()/2,
            f'£{row["lifetime_value"]:,.0f}  ({row["total_orders"]} orders)',
            va='center', fontsize=9)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'£{x:,.0f}'))
ax.set_title('Top 10 Customers by Lifetime Value', fontsize=13, fontweight='bold')
ax.set_xlim(0, q3['lifetime_value'].max() * 1.35)
plt.tight_layout()
plt.savefig('q3_top_customers.png', dpi=150, bbox_inches='tight')
plt.show()

### Q4 — Regional Performance with CASE WHEN Segmentation
*CASE WHEN, GROUP BY multiple columns, conditional aggregation*

In [ ]:
sql_q4 = '''
SELECT
    c.region,
    COUNT(DISTINCT c.customer_id)            AS customers,
    COUNT(DISTINCT o.order_id)               AS orders,
    ROUND(SUM(oi.sales), 2)                  AS revenue,
    ROUND(SUM(oi.profit), 2)                 AS profit,
    ROUND(SUM(oi.sales) / COUNT(DISTINCT o.order_id), 2) AS avg_order_value,
    CASE
        WHEN SUM(oi.profit)*100.0/SUM(oi.sales) >= 25 THEN 'High'
        WHEN SUM(oi.profit)*100.0/SUM(oi.sales) >= 15 THEN 'Medium'
        ELSE 'Low'
    END AS margin_band
FROM customers c
JOIN orders      o  ON c.customer_id  = o.customer_id
JOIN order_items oi ON o.order_id     = oi.order_id
GROUP BY c.region
ORDER BY revenue DESC
'''
q4 = query(sql_q4, 'Regional Performance')

band_colours = {'High': GREEN, 'Medium': BLUE, 'Low': RED}
colours = [band_colours[b] for b in q4['margin_band']]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].bar(q4['region'], q4['revenue'], color=colours, edgecolor='white')
axes[0].set_xticklabels(q4['region'], rotation=30, ha='right')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'£{x/1000:.0f}k'))
axes[0].set_title('Revenue by Region (colour = margin band)', fontweight='bold')

axes[1].bar(q4['region'], q4['avg_order_value'], color=NAVY, edgecolor='white')
axes[1].set_xticklabels(q4['region'], rotation=30, ha='right')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'£{x:,.0f}'))
axes[1].set_title('Average Order Value by Region', fontweight='bold')

from matplotlib.patches import Patch
legend = [Patch(color=c, label=l) for l, c in band_colours.items()]
axes[0].legend(handles=legend, title='Margin Band')
plt.suptitle('UK Regional Sales Performance', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('q4_regional_performance.png', dpi=150, bbox_inches='tight')
plt.show()

### Q5 — Products with Above-Average Sales (Subquery)
*Scalar subquery in WHERE clause*

In [ ]:
sql_q5 = '''
SELECT
    p.product_name,
    p.category,
    p.sub_category,
    ROUND(SUM(oi.sales), 2)   AS total_sales,
    SUM(oi.quantity)          AS units_sold,
    ROUND(SUM(oi.profit), 2)  AS total_profit
FROM products p
JOIN order_items oi ON p.product_id = oi.product_id
GROUP BY p.product_id
HAVING total_sales > (
    SELECT AVG(product_sales)
    FROM (
        SELECT SUM(sales) AS product_sales
        FROM order_items
        GROUP BY product_id
    )
)
ORDER BY total_sales DESC
'''
q5 = query(sql_q5, 'Above-Average Products')

fig, ax = plt.subplots(figsize=(12, 6))
cat_colours = {'Technology': NAVY, 'Furniture': BLUE, 'Office Supplies': LIGHT}
colours = [cat_colours.get(c, NAVY) for c in q5['category']]
ax.barh(q5['product_name'][::-1], q5['total_sales'][::-1], color=colours[::-1], edgecolor='white')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'£{x:,.0f}'))
ax.set_title('Products with Above-Average Total Sales', fontsize=13, fontweight='bold')
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=c, label=l) for l,c in cat_colours.items()])
plt.tight_layout()
plt.savefig('q5_top_products.png', dpi=150, bbox_inches='tight')
plt.show()

### Q6 — Monthly Revenue with Running Total (Window Function)
*`SUM() OVER` cumulative window function — one of the most valued SQL skills*

In [ ]:
sql_q6 = '''
WITH monthly AS (
    SELECT
        strftime('%Y-%m', o.order_date)  AS month,
        ROUND(SUM(oi.sales), 2)          AS monthly_revenue
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    GROUP BY month
)
SELECT
    month,
    monthly_revenue,
    ROUND(
        SUM(monthly_revenue) OVER (ORDER BY month
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW),
        2
    ) AS running_total
FROM monthly
ORDER BY month
'''
q6 = query(sql_q6)

fig, ax1 = plt.subplots(figsize=(14, 5))
ax2 = ax1.twinx()

x = range(len(q6))
ax1.bar(x, q6['monthly_revenue'], color=BLUE, alpha=0.7, label='Monthly Revenue')
ax2.plot(x, q6['running_total'], color=RED, linewidth=2.5, label='Running Total')

tick_every = 3
ax1.set_xticks(list(x)[::tick_every])
ax1.set_xticklabels(q6['month'].iloc[::tick_every], rotation=45, ha='right')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'£{v/1000:.0f}k'))
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'£{v/1000:.0f}k'))
ax1.set_ylabel('Monthly Revenue', color=BLUE)
ax2.set_ylabel('Cumulative Revenue', color=RED)
ax1.set_title('Monthly Revenue & Cumulative Running Total (2021–2023)',
              fontsize=13, fontweight='bold')
lines1, l1 = ax1.get_legend_handles_labels()
lines2, l2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2, l1+l2, loc='upper left')
plt.tight_layout()
plt.savefig('q6_running_total.png', dpi=150, bbox_inches='tight')
plt.show()

### Q7 — Customer Ranking by Region Using RANK()
*Window function with PARTITION BY — ranks reset per region*

In [ ]:
sql_q7 = '''
WITH customer_revenue AS (
    SELECT
        c.customer_id,
        c.first_name || ' ' || c.last_name  AS customer_name,
        c.region,
        ROUND(SUM(oi.sales), 2)              AS revenue
    FROM customers c
    JOIN orders      o  ON c.customer_id = o.customer_id
    JOIN order_items oi ON o.order_id    = oi.order_id
    GROUP BY c.customer_id
)
SELECT
    region,
    customer_name,
    revenue,
    RANK() OVER (PARTITION BY region ORDER BY revenue DESC) AS rank_in_region
FROM customer_revenue
ORDER BY region, rank_in_region
'''
q7 = query(sql_q7)

# Show top 1 per region
top_per_region = q7[q7['rank_in_region'] == 1].sort_values('revenue', ascending=False)
print('=== Top Customer per Region ===')
print(top_per_region[['region','customer_name','revenue']].to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 5))
ax.barh(top_per_region['region'][::-1], top_per_region['revenue'][::-1],
        color=NAVY, edgecolor='white')
for i, (_, row) in enumerate(top_per_region[::-1].iterrows()):
    ax.text(row['revenue'] + 100, i, f"{row['customer_name']}  £{row['revenue']:,.0f}",
            va='center', fontsize=9)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'£{x:,.0f}'))
ax.set_xlim(0, top_per_region['revenue'].max() * 1.4)
ax.set_title('Top-Ranked Customer per Region (RANK OVER PARTITION BY)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('q7_rank_by_region.png', dpi=150, bbox_inches='tight')
plt.show()

### Q8 — Month-over-Month Revenue Growth Using a CTE
*CTE + LAG() window function — calculates % change vs prior month*

In [ ]:
sql_q8 = '''
WITH monthly_rev AS (
    SELECT
        strftime('%Y-%m', o.order_date)  AS month,
        ROUND(SUM(oi.sales), 2)          AS revenue
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    GROUP BY month
),
with_lag AS (
    SELECT
        month,
        revenue,
        LAG(revenue) OVER (ORDER BY month) AS prev_month_revenue
    FROM monthly_rev
)
SELECT
    month,
    revenue,
    prev_month_revenue,
    ROUND(
        (revenue - prev_month_revenue) * 100.0 / prev_month_revenue,
        1
    ) AS mom_growth_pct
FROM with_lag
WHERE prev_month_revenue IS NOT NULL
ORDER BY month
'''
q8 = query(sql_q8)

colours = [GREEN if v >= 0 else RED for v in q8['mom_growth_pct']]
x = range(len(q8))

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(x, q8['mom_growth_pct'], color=colours, edgecolor='white')
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xticks(list(x)[::3])
ax.set_xticklabels(q8['month'].iloc[::3], rotation=45, ha='right')
ax.set_ylabel('Month-over-Month Growth %')
ax.set_title('Month-over-Month Revenue Growth — UK Retail 2021–2023\n(CTE + LAG window function)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('q8_mom_growth.png', dpi=150, bbox_inches='tight')
plt.show()

positive = (q8['mom_growth_pct'] > 0).sum()
print(f'Positive growth months: {positive} / {len(q8)}')

### Q9 — Best-Selling Product in Each Category (Correlated Subquery)
*Subquery that references the outer query — a classic interview question*

In [ ]:
sql_q9 = '''
SELECT
    p.category,
    p.product_name,
    ROUND(SUM(oi.sales), 2)  AS total_sales,
    SUM(oi.quantity)         AS units_sold
FROM products p
JOIN order_items oi ON p.product_id = oi.product_id
GROUP BY p.product_id
HAVING total_sales = (
    SELECT MAX(sub_sales)
    FROM (
        SELECT SUM(oi2.sales) AS sub_sales
        FROM products p2
        JOIN order_items oi2 ON p2.product_id = oi2.product_id
        WHERE p2.category = p.category
        GROUP BY p2.product_id
    )
)
ORDER BY total_sales DESC
'''
q9 = query(sql_q9, 'Best Product per Category')

### Q10 — Customer Segment Profitability & Discount Impact
*Multi-level GROUP BY with conditional aggregation and HAVING filter*

In [ ]:
sql_q10 = '''
SELECT
    c.segment,
    p.category,
    COUNT(DISTINCT o.order_id)                              AS orders,
    ROUND(SUM(oi.sales), 2)                                 AS revenue,
    ROUND(SUM(oi.profit), 2)                                AS profit,
    ROUND(AVG(oi.discount) * 100, 1)                        AS avg_discount_pct,
    ROUND(SUM(oi.profit) * 100.0 / SUM(oi.sales), 1)       AS margin_pct,
    ROUND(SUM(oi.sales) / COUNT(DISTINCT o.order_id), 2)    AS avg_order_value
FROM customers    c
JOIN orders       o  ON c.customer_id = o.customer_id
JOIN order_items  oi ON o.order_id    = oi.order_id
JOIN products     p  ON oi.product_id = p.product_id
GROUP BY c.segment, p.category
HAVING orders > 10
ORDER BY profit DESC
'''
q10 = query(sql_q10, 'Segment × Category Profitability')

pivot = q10.pivot_table(index='segment', columns='category', values='margin_pct')

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='Blues', ax=ax,
            linewidths=0.5, cbar_kws={'label': 'Profit Margin %'})
ax.set_title('Profit Margin % by Customer Segment & Product Category',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Product Category')
ax.set_ylabel('Customer Segment')
plt.tight_layout()
plt.savefig('q10_segment_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Closing the Connection

In [ ]:
conn.close()
print('Database connection closed.')

## 6. Key Findings & Conclusions

---

### SQL Techniques Used

| Query | Technique |
|---|---|
| Q1 - Annual Revenue | `JOIN`, `GROUP BY`, `strftime` date functions |
| Q2 - Category Performance | Multi-table `JOIN`, computed margin column |
| Q3 - Top Customers | 3-table `JOIN`, string concatenation, `LIMIT` |
| Q4 — Regional Analysis | `CASE WHEN` conditional classification |
| Q5 — Above-Average Products | Scalar subquery in `HAVING` clause |
| Q6 — Running Total | `SUM() OVER` cumulative window function + CTE |
| Q7 — Regional Rankings | `RANK() OVER (PARTITION BY)` window function |
| Q8 — MoM Growth | `LAG()` window function + chained CTEs |
| Q9 — Best per Category | Correlated subquery |
| Q10 - Segment Heatmap | Multi-level `GROUP BY`, `HAVING`, 4-table `JOIN` |

### Business Insights

- **Technology** drives the highest revenue despite lower profit margins, volume compensates for thin margins
- **Office Supplies** has the highest margin percentage, making it the most profitable category relative to revenue
- **Corporate** segment customers have higher average order values than Consumer customers
- **London** and the **South East** generate the most revenue, but some northern regions show comparable margin bands
- Revenue shows clear **seasonal patterns** with growth months clustering in Q3/Q4
- Discounts above 20% consistently correlate with reduced profitability across all segments

### Next Steps / Future Work

- Connect to a real SQL Server or PostgreSQL database and replicate these queries
- Build a Power BI dashboard consuming the same SQL queries
- Apply cohort analysis to measure customer retention over time
- Use Python `sqlalchemy` for production-grade database connections
- Add stored procedures and views to the schema

---
**Database:** SQLite (via Python `sqlite3` standard library)  
**Data:** Synthetically generated using realistic UK retail parameters